In [1]:
%run /home/hadoop/agrim_cdp/common_utils/db.ipynb

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import logging
from datetime import datetime
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col, current_timestamp
from pyspark.sql.functions import to_date

In [3]:
# ---- Configurations ----
BUCKET = 'agrim-cdp'
PREFIX = 'data-landing/callyzer/'
BATCH_SIZE = 300
TABLE_NAME = 'cdp_raw_db.callyzer_call_logs_raw'
S3_REGION = 'ap-south-1'

In [4]:
import os
os.environ["SPARK_HOME"] = "/home/hadoop/.local/lib/python3.9/site-packages/pyspark"  # Or wherever your Spark is
os.environ["PATH"] = os.environ["SPARK_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# ---- Initialize Spark & boto3 ----
spark = SparkSession.builder.appName("callyzer_batch_load") \
    .config("spark.jars", "/home/hadoop/agrim_cdp/common_jars/postgresql-42.2.24.jar") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
s3 = boto3.client('s3', region_name=S3_REGION)
logger = logging.getLogger('CALLYZER_DATA_LOAD')
logging.basicConfig(level=logging.INFO)

:: loading settings :: url = jar:file:/home/hadoop/.local/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hadoop/.ivy2/cache
The jars for the packages stored in: /home/hadoop/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-61bd2083-10d6-4dd6-a128-636da45801e0;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in spark-list
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in spark-list


	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in spark-list
:: resolution report :: resolve 213ms :: artifacts dl 9ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from spark-list in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from spark-list in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from spark-list in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-61bd2083-10d6-4dd6-a128-636da45801e0
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/6ms)


25/07/16 06:40:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/07/16 06:40:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/07/16 06:40:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [6]:
spark._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")

In [7]:
# ---- Step 1: List files ----
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=BATCH_SIZE)
files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.txt')]

In [8]:
if not files:
    logger.info("No JSON files found.")
    exit(0)

In [9]:
logger.info(f"Processing {len(files)} files.")

INFO:CALLYZER_DATA_LOAD:Processing 98 files.


In [10]:
# ---- Step 2: Read files into Spark DataFrame ----
file_paths = [f"s3a://{BUCKET}/{key}" for key in files]
raw_df = spark.read.option("multiline", "true").json(file_paths)

25/07/16 06:40:13 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


25/07/16 06:40:14 WARN CredentialsLegacyConfigLocationProvider: Found the legacy config profiles file at [/home/hadoop/.aws/config]. Please move it to the latest default location [~/.aws/credentials].


In [11]:
# ---- Step 3: Flatten nested structure ----
flattened_df = (
    raw_df
    .withColumn("log", explode("logs"))
    .select(
        col("employeeName").alias("employee_name"),
        col("employeeCode").alias("employee_code"),
        col("countryCode").alias("employee_country_code"),
        col("employeeNumber").alias("employee_number"),
        col("employeeTags").alias("employee_tags"),
        col("log.id").alias("id"),
        col("log.name").alias("name"),
        col("log.countryCode").alias("country_code"),
        col("log.number"),
        col("log.duration"),
        col("log.callType").alias("call_type"),
        col("log.callTime").alias("call_time"),
        col("log.callTimeStnd").alias("call_time_stnd"),
        col("log.note").alias("note"),
        col("log.recordingURL").alias("recording_url"),
        col("log.crmStatus").alias("crm_status"),
        col("log.reminderTime").alias("reminder_time"),
        col("log.reminderTimeStnd").alias("reminder_time_stnd"),
        col("log.createdDate").alias("created_date"),
        col("log.modifiedDate").alias("modified_date"),
        col("log.createdDateStnd").alias("created_date_stnd"),
        col("log.modifiedDateStnd").alias("modified_date_stnd"),
        col("log.callTimeStnd").substr(1, 10).alias("call_date"),
        current_timestamp().alias("create_timestamp")
    )
)
flattened_df = flattened_df.withColumn("call_date", to_date("call_date"))
logger.info(f"Total rows to insert: {flattened_df.count()}")

INFO:CALLYZER_DATA_LOAD:Total rows to insert: 129


In [12]:
RDS_HOST, RDS_DBNM, RDS_USER, RDS_PASSWORD, RDS_PORT = get_rds_credentials()

In [13]:
# ---- Step 4: Write to RDS using JDBC ----
flattened_df.write \
    .format("jdbc") \
    .option("url", f'jdbc:postgresql://{RDS_HOST}:{RDS_PORT}/{RDS_DBNM}') \
    .option("dbtable", TABLE_NAME) \
    .option("user", RDS_USER) \
    .option("password", RDS_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [14]:
logger.info("Data written to RDS.")

INFO:CALLYZER_DATA_LOAD:Data written to RDS.


In [15]:
# ---- Step 5: Delete files from S3 ----
for key in files:
    try:
        s3.delete_object(Bucket=BUCKET, Key=key)
        logger.info(f"Deleted: s3://{BUCKET}/{key}")
    except Exception as e:
        logger.warning(f"Failed to delete {key}: {str(e)}")

INFO:CALLYZER_DATA_LOAD:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-07-16/1752647715.684313323832287965.txt


INFO:CALLYZER_DATA_LOAD:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-07-16/1752647716.028541846536725168.txt


INFO:CALLYZER_DATA_LOAD:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-07-16/1752647716.387908716040756660.txt


INFO:CALLYZER_DATA_LOAD:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-07-16/1752647717.284945724131957713.txt


INFO:CALLYZER_DATA_LOAD:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-07-16/1752647727.227607729314034088.txt


...

INFO:CALLYZER_DATA_LOAD:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-07-16/1752647991.902035210106455931.txt


INFO:CALLYZER_DATA_LOAD:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-07-16/1752647993.00212820471833645.txt


INFO:CALLYZER_DATA_LOAD:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-07-16/1752648000.483626822806469231.txt


INFO:CALLYZER_DATA_LOAD:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-07-16/1752648001.160719940453664551.txt


INFO:CALLYZER_DATA_LOAD:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-07-16/1752648010.319831431860725898.txt


In [16]:
logger.info("Batch job completed successfully.")

INFO:CALLYZER_DATA_LOAD:Batch job completed successfully.
